# Solar + Wind Estimation for 250 / 290 / 350 Panels

This notebook loads the cleaned dataset, scales solar generation for different PV array sizes, and identifies the lowest 15-day generation windows.

In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path('Power_conso_farm_projet_good_calcul_anne_cleaned.csv')
df = pd.read_csv(csv_path, sep=';', parse_dates=['local_time'])
df.head()

,local_time,wind_electricity_kW,wind_speed_m_s,solar_electricity_kW,solar_electricity_W,consommation_kW
0,01/01/2019 01:00,27.862,6505,0.0,0,30
1,01/01/2019 02:00,28.532,6560,0.0,0,30
2,01/01/2019 03:00,26.972,6430,0.0,0,30
3,01/01/2019 04:00,22.973,6076,0.0,0,30
4,01/01/2019 05:00,20.624,5862,0.0,0,33


In [ ]:
numeric_cols = [
    'wind_electricity_kW',
    'wind_speed_m_s',
    'solar_electricity_kW',
    'solar_electricity_W',
    'consommation_kW',
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')

# Convert raw wind electricity values from W to kW
# The source file stores wind generation in watts, so we normalize it here.
df['wind_electricity_kW'] = df['wind_electricity_kW'] / 1000

# Convert 'local_time' to datetime objects
df['local_time'] = pd.to_datetime(df['local_time'], format='%d/%m/%Y %H:%M', errors='coerce')

df['hour'] = df['local_time'].dt.hour
df['date'] = df['local_time'].dt.date
df.dtypes

,0
local_time,datetime64[ns]
wind_electricity_kW,float64
wind_speed_m_s,int64
solar_electricity_kW,float64
solar_electricity_W,int64
consommation_kW,int64
hour,int32
date,object


## Scaling solar production

We use `solar_electricity_kW` as the baseline solar profile. If this dataset already corresponds to a 290-panel system, the correct scaling for 250 and 350 panels is `250/290` and `350/290`.
If the column is instead a per-panel baseline, the direct multiplication by each panel count is appropriate.

In [ ]:
panel_counts = [250, 290, 350]
for n in panel_counts:
    df[f'solar_{n}_kW'] = df['solar_electricity_kW'] * n
    df[f'total_{n}_kW'] = df['wind_electricity_kW'] + df[f'solar_{n}_kW']

for n in panel_counts:
    df[f'solar_{n}_from_290_kW'] = df['solar_electricity_kW'] * (n / 290)
    df[f'total_{n}_from_290_kW'] = df['wind_electricity_kW'] + df[f'solar_{n}_from_290_kW']

df[[f'solar_{n}_kW' for n in panel_counts]].head()

,solar_250_kW,solar_290_kW,solar_350_kW
0,0.0,0.0,0.0
1,0.0,0.0,0.0
2,0.0,0.0,0.0
3,0.0,0.0,0.0
4,0.0,0.0,0.0


In [ ]:
df_daily = df.drop(columns=['date']).set_index('local_time').resample('D').sum()
df_daily = df_daily.rename_axis('date')
df_daily[[
    'wind_electricity_kW',
    'solar_250_kW',
    'solar_290_kW',
    'solar_350_kW',
    'total_250_kW',
    'total_290_kW',
    'total_350_kW',
    'consommation_kW'
]].head()

,wind_electricity_kW,solar_250_kW,solar_290_kW,solar_350_kW,total_250_kW,total_290_kW,total_350_kW,consommation_kW
date,,,,,,,,
2019-01-01,0.425331,46253.50,53654.06,64754.90,46253.925331,53654.485331,64755.325331,873
2019-01-02,1.150316,51660.25,59925.89,72324.35,51661.400316,59927.040316,72325.500316,903
2019-01-03,0.667998,55998.00,64957.68,78397.20,55998.667998,64958.347998,78397.867998,903
2019-01-04,0.511497,50627.25,58727.61,70878.15,50627.761497,58728.121497,70878.661497,903
2019-01-05,1.223786,53649.25,62233.13,75108.95,53650.473786,62234.353786,75110.173786,903


In [ ]:
# Calculate rolling sums for columns generated by direct multiplication
for n in panel_counts:
    df_daily[f'rolling_15_total_{n}'] = df_daily[f'total_{n}_kW'].rolling(15).sum()

# Calculate rolling sums for columns generated by scaling from 290
# These are generated for all panel counts, including 290 as the reference case
for n in panel_counts:
    df_daily[f'rolling_15_total_{n}_from_290'] = df_daily[f'total_{n}_from_290_kW'].rolling(15).sum()

low_windows = []
# Process the 'multiply_by_count' method
for n in panel_counts:
    col = f'rolling_15_total_{n}'
    low = df_daily[[col]].nsmallest(5, col).copy()
    low['panel_count'] = n
    low['window_end'] = low.index
    low['window_start'] = low.index - pd.Timedelta(days=14)
    low['method'] = 'multiply_by_count'
    low = low.rename(columns={col: 'energy_15d_kWh'})
    low_windows.append(low.reset_index(drop=True))

# Process the 'scale_from_290' method
for n in panel_counts:
    col = f'rolling_15_total_{n}_from_290'
    low = df_daily[[col]].nsmallest(5, col).copy()
    low['panel_count'] = n
    low['window_end'] = low.index
    low['window_start'] = low.index - pd.Timedelta(days=14)
    low['method'] = 'scale_from_290'
    low = low.rename(columns={col: 'energy_15d_kWh'})
    low_windows.append(low.reset_index(drop=True))

low_df = pd.concat(low_windows, ignore_index=True)
low_df.sort_values(['panel_count', 'method', 'energy_15d_kWh']).head(20)

,energy_15d_kWh,panel_count,window_end,window_start,method
0,439089.877945,250,2019-12-17,2019-12-03,multiply_by_count
1,449652.496578,250,2019-12-18,2019-12-04,multiply_by_count
2,460640.696315,250,2019-12-25,2019-12-11,multiply_by_count
3,460663.297937,250,2019-12-26,2019-12-12,multiply_by_count
4,461744.056200,250,2019-12-16,2019-12-02,multiply_by_count
15,1523.697773,250,2019-12-17,2019-12-03,scale_from_290
16,1560.238819,250,2019-12-18,2019-12-04,scale_from_290
17,1600.321315,250,2019-12-25,2019-12-11,scale_from_290
18,1600.500523,250,2019-12-26,2019-12-12,scale_from_290
19,1601.744131,250,2019-12-16,2019-12-02,scale_from_290


In [ ]:
low_df.to_csv('low_15_day_windows.csv', sep=';', index=False)
low_df.shape

(25, 5)

## Dimensionnement des panneaux solaires

Nous allons maintenant calculer le nombre de panneaux supplémentaires nécessaires pour couvrir la consommation pendant la période de production la plus faible, qui a été identifiée comme étant du **2019-12-03 au 2019-12-17**.

In [ ]:
import pandas as pd

# Définir la période de faible production
worst_period_start = pd.to_datetime('2019-12-03')
worst_period_end = pd.to_datetime('2019-12-17')

# Filtrer df_daily pour cette période
df_worst_period = df_daily[(df_daily.index >= worst_period_start) & (df_daily.index <= worst_period_end)]

# Calculer la consommation totale pour cette période
total_consumption_worst_period_kWh = df_worst_period['consommation_kW'].sum()
print(f"Consommation totale pendant la période la plus faible (15 jours): {total_consumption_worst_period_kWh:.2f} kWh")

# Récupérer la production totale d'énergie (vent + solaire) pour 290 panneaux (méthode 'multiply_by_count')
# La colonne 'total_290_kW' dans df_daily représente la somme du vent et du solaire pour 290 panneaux (méthode multiply_by_count)
total_generation_290_panels_worst_period_kWh = df_worst_period['total_290_kW'].sum()
print(f"Production totale (290 panneaux, méthode multiply_by_count) pendant la période la plus faible: {total_generation_290_panels_worst_period_kWh:.2f} kWh")

# Calculer le déficit énergétique
energy_deficit_kWh = total_consumption_worst_period_kWh - total_generation_290_panels_worst_period_kWh
print(f"Déficit énergétique avec 290 panneaux: {energy_deficit_kWh:.2f} kWh")

# Calculer la production solaire par panneau pour cette période
# Nous utilisons la colonne 'solar_290_kW' qui représente la production solaire de 290 panneaux (méthode multiply_by_count)
# Donc, la production solaire par panneau est (solar_290_kW / 290)
daily_solar_per_panel_worst_period = df_worst_period['solar_290_kW'] / 290
total_solar_per_panel_worst_period_kWh = daily_solar_per_panel_worst_period.sum()
print(f"Production solaire totale par panneau pendant la période la plus faible: {total_solar_per_panel_worst_period_kWh:.2f} kWh")

# Déterminer les panneaux supplémentaires nécessaires
if energy_deficit_kWh > 0:
    # Les panneaux supplémentaires ne contribuent qu'à la production solaire (la production éolienne est supposée constante)
    additional_solar_needed_kWh = energy_deficit_kWh
    if total_solar_per_panel_worst_period_kWh > 0:
        additional_panels_needed = additional_solar_needed_kWh / total_solar_per_panel_worst_period_kWh
        current_panels = 290 # Base number of panels for this calculation
        total_panels_required = current_panels + additional_panels_needed
        print(f"Panneaux supplémentaires nécessaires pour couvrir le déficit: {additional_panels_needed:.2f}")
        print(f"Nombre total de panneaux recommandés: {total_panels_required:.2f} (arrondi à {int(total_panels_required)+1} si on ne peut pas avoir de fraction de panneau)")
    else:
        print("Impossible de calculer les panneaux supplémentaires car la production solaire par panneau est nulle pour cette période.")
else:
    print("Pas de déficit énergétique, aucun panneau supplémentaire n'est nécessaire.")

Consommation totale pendant la période la plus faible (15 jours): 13545.00 kWh
Production totale (290 panneaux, méthode multiply_by_count) pendant la période la plus faible: 509342.72 kWh
Déficit énergétique avec 290 panneaux: -495797.72 kWh
Production solaire totale par panneau pendant la période la plus faible: 1756.32 kWh
Pas de déficit énergétique, aucun panneau supplémentaire n'est nécessaire.


In [ ]:
import math
import pandas as pd

# Donnees et constantes pour le dimensionnement chronologique annuel.
df_sizing = pd.read_csv(csv_path, sep=';', parse_dates=['local_time'], dayfirst=True)
numeric_cols = [
    'wind_electricity_kW',
    'solar_electricity_kW',
    'solar_electricity_W',
    'consommation_kW',
]
for col in numeric_cols:
    df_sizing[col] = pd.to_numeric(df_sizing[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')

reference_panel_count = 290
battery_capacity_kWh = 2.40
min_soc = 0.20
max_soc = 0.90
usable_fraction = max_soc - min_soc
usable_energy_per_battery_kWh = battery_capacity_kWh * usable_fraction

panel_unit_cost_EUR = 139.66
battery_unit_cost_EUR = 449.99

worst_period_start = pd.to_datetime('2019-12-03')
worst_period_end = pd.to_datetime('2019-12-17 23:59:59')

df_sizing['solar_per_panel_kWh'] = df_sizing['solar_electricity_kW'] / reference_panel_count
critical_period_mask = (
    (df_sizing['local_time'] >= worst_period_start)
    & (df_sizing['local_time'] <= worst_period_end)
)
df_critical_period = df_sizing.loc[critical_period_mask].copy()

period_consumption_kWh = df_critical_period['consommation_kW'].sum()
period_wind_kWh = df_critical_period['wind_electricity_kW'].sum()
period_solar_per_panel_kWh = df_critical_period['solar_per_panel_kWh'].sum()
min_panels_for_period_balance = math.ceil((period_consumption_kWh - period_wind_kWh) / period_solar_per_panel_kWh)

print(f"Periode critique: {worst_period_start.date()} au {worst_period_end.date()}")
print(f"Nombre d'heures: {len(df_critical_period)}")
print(f"Consommation sur la periode: {period_consumption_kWh:.2f} kWh")
print(f"Production eolienne sur la periode: {period_wind_kWh:.2f} kWh")
print(f"Production solaire par panneau sur la periode: {period_solar_per_panel_kWh:.2f} kWh/panneau")
print(f"Minimum de panneaux pour un bilan positif sur la periode: {min_panels_for_period_balance}")
print(f"Energie utilisable par batterie: {usable_energy_per_battery_kWh:.2f} kWh")

In [ ]:
# Dimensionnement economique panneaux/batteries sur la periode critique.
# Hypothese: les batteries commencent la periode critique chargees a 90%.
panel_search_min = 0
panel_search_max = 5000

def required_usable_capacity_kWh(hourly_balance_kWh):
    cumulative = hourly_balance_kWh.cumsum()
    cumulative = pd.concat([pd.Series([0.0]), cumulative.reset_index(drop=True)], ignore_index=True)
    running_peak = cumulative.cummax()
    drawdown = running_peak - cumulative
    return drawdown.max()

records = []
for panel_count in range(panel_search_min, panel_search_max + 1):
    hourly_solar_kWh = df_critical_period['solar_per_panel_kWh'] * panel_count
    hourly_balance_kWh = df_critical_period['wind_electricity_kW'] + hourly_solar_kWh - df_critical_period['consommation_kW']
    period_balance_kWh = hourly_balance_kWh.sum()
    required_usable_kWh = required_usable_capacity_kWh(hourly_balance_kWh)
    battery_count = math.ceil(required_usable_kWh / usable_energy_per_battery_kWh)
    records.append({
        'panels': panel_count,
        'period_balance_kWh': period_balance_kWh,
        'required_usable_battery_kWh': required_usable_kWh,
        'batteries': battery_count,
        'nominal_battery_capacity_kWh': battery_count * battery_capacity_kWh,
        'usable_battery_capacity_kWh': battery_count * usable_energy_per_battery_kWh,
        'panel_cost_EUR': panel_count * panel_unit_cost_EUR,
        'battery_cost_EUR': battery_count * battery_unit_cost_EUR,
        'total_cost_EUR': panel_count * panel_unit_cost_EUR + battery_count * battery_unit_cost_EUR,
    })

dimensioning_df = pd.DataFrame(records)
dimensioning_csv_path = 'dimensionnement_panneaux_batteries_periode_critique.csv'
dimensioning_df.to_csv(dimensioning_csv_path, sep=';', index=False)

selected = dimensioning_df.loc[dimensioning_df['total_cost_EUR'].idxmin()]
selected_panel_count = int(selected['panels'])
selected_battery_count = int(selected['batteries'])

display(dimensioning_df.loc[
    dimensioning_df['panels'].isin([0, 250, 290, 350, 647, 1000, selected_panel_count, 1500, 2000]),
    ['panels', 'period_balance_kWh', 'required_usable_battery_kWh', 'batteries', 'nominal_battery_capacity_kWh', 'total_cost_EUR']
])
print(f"Solution economique retenue: {selected_panel_count} panneaux et {selected_battery_count} batteries")
print(f"Capacite nominale batterie: {selected['nominal_battery_capacity_kWh']:.2f} kWh")
print(f"Capacite utilisable batterie: {selected['usable_battery_capacity_kWh']:.2f} kWh")
print(f"Cout panneaux: {selected['panel_cost_EUR']:.2f} EUR")
print(f"Cout batteries: {selected['battery_cost_EUR']:.2f} EUR")
print(f"Cout total: {selected['total_cost_EUR']:.2f} EUR")
print(f"CSV dimensionnement: {dimensioning_csv_path}")

In [ ]:
# Simulation du SOC sur la periode critique pour la solution retenue.
def simulate_battery(panel_count, battery_count, initial_soc=max_soc, export_path=None):
    df_sim = df_critical_period.copy()
    solar_col = f'solar_{panel_count}_panneaux_kWh'
    df_sim[solar_col] = df_sim['solar_per_panel_kWh'] * panel_count
    df_sim['bilan_horaire_kWh'] = (
        df_sim['wind_electricity_kW']
        + df_sim[solar_col]
        - df_sim['consommation_kW']
    )

    total_capacity_kWh = battery_count * battery_capacity_kWh
    min_energy_kWh = min_soc * total_capacity_kWh
    max_energy_kWh = max_soc * total_capacity_kWh
    energy_kWh = min(max(initial_soc, min_soc), max_soc) * total_capacity_kWh

    stored_energy = []
    soc_values = []
    unserved_energy = []
    spilled_energy = []

    for balance_kWh in df_sim['bilan_horaire_kWh']:
        next_energy_kWh = energy_kWh + balance_kWh
        unserved_kWh = 0.0
        spilled_kWh = 0.0

        if next_energy_kWh > max_energy_kWh:
            spilled_kWh = next_energy_kWh - max_energy_kWh
            next_energy_kWh = max_energy_kWh
        elif next_energy_kWh < min_energy_kWh:
            unserved_kWh = min_energy_kWh - next_energy_kWh
            next_energy_kWh = min_energy_kWh

        energy_kWh = next_energy_kWh
        stored_energy.append(energy_kWh)
        soc_values.append(100 * energy_kWh / total_capacity_kWh)
        unserved_energy.append(unserved_kWh)
        spilled_energy.append(spilled_kWh)

    df_sim['energie_stockee_batteries_kWh'] = stored_energy
    df_sim['etat_batterie_pct'] = soc_values
    df_sim['energie_non_servie_kWh'] = unserved_energy
    df_sim['energie_ecretee_kWh'] = spilled_energy

    summary = {
        'initial_soc_pct': initial_soc * 100,
        'min_soc_pct': df_sim['etat_batterie_pct'].min(),
        'max_soc_pct': df_sim['etat_batterie_pct'].max(),
        'unserved_kWh': df_sim['energie_non_servie_kWh'].sum(),
        'spilled_kWh': df_sim['energie_ecretee_kWh'].sum(),
    }

    if export_path is not None:
        df_sim.to_csv(export_path, sep=';', index=False)

    return df_sim, summary

battery_csv_path = 'periode_critique_with_battery.csv'
df_battery, selected_summary = simulate_battery(
    selected_panel_count,
    selected_battery_count,
    initial_soc=max_soc,
    export_path=battery_csv_path,
)

# Sensibilite a l'etat initial: a 20%, il n'y a aucune reserve utilisable au debut.
initial_soc_sensitivity = pd.DataFrame([
    simulate_battery(selected_panel_count, selected_battery_count, initial_soc=soc)[1]
    for soc in [0.90, 0.50, 0.20]
])

display(initial_soc_sensitivity)
print(f"CSV enrichi: {battery_csv_path}")

In [ ]:
# Simulation annuelle avec la solution economique dimensionnee sur la periode critique.
annual_panel_count = selected_panel_count
annual_battery_count = selected_battery_count

def simulate_storage(df_input, panel_count, battery_count, initial_soc=max_soc):
    df_sim = df_input.copy()
    solar_col = f'solar_{panel_count}_panneaux_kWh'
    df_sim[solar_col] = df_sim['solar_per_panel_kWh'] * panel_count
    df_sim['generation_totale_kWh'] = df_sim['wind_electricity_kW'] + df_sim[solar_col]
    df_sim['bilan_horaire_kWh'] = df_sim['generation_totale_kWh'] - df_sim['consommation_kW']

    total_capacity_kWh = battery_count * battery_capacity_kWh
    min_energy_kWh = min_soc * total_capacity_kWh
    max_energy_kWh = max_soc * total_capacity_kWh
    energy_kWh = min(max(initial_soc, min_soc), max_soc) * total_capacity_kWh

    stored_energy = []
    soc_values = []
    unserved_energy = []
    spilled_energy = []

    for balance_kWh in df_sim['bilan_horaire_kWh']:
        next_energy_kWh = energy_kWh + balance_kWh
        unserved_kWh = 0.0
        spilled_kWh = 0.0

        if next_energy_kWh > max_energy_kWh:
            spilled_kWh = next_energy_kWh - max_energy_kWh
            next_energy_kWh = max_energy_kWh
        elif next_energy_kWh < min_energy_kWh:
            unserved_kWh = min_energy_kWh - next_energy_kWh
            next_energy_kWh = min_energy_kWh

        energy_kWh = next_energy_kWh
        stored_energy.append(energy_kWh)
        soc_values.append(100 * energy_kWh / total_capacity_kWh)
        unserved_energy.append(unserved_kWh)
        spilled_energy.append(spilled_kWh)

    df_sim['energie_stockee_batteries_kWh'] = stored_energy
    df_sim['etat_batterie_pct'] = soc_values
    df_sim['energie_non_servie_kWh'] = unserved_energy
    df_sim['energie_ecretee_kWh'] = spilled_energy

    summary = {
        'panels': panel_count,
        'batteries': battery_count,
        'consommation_kWh': df_sim['consommation_kW'].sum(),
        'wind_kWh': df_sim['wind_electricity_kW'].sum(),
        'solar_kWh': df_sim[solar_col].sum(),
        'generation_kWh': df_sim['generation_totale_kWh'].sum(),
        'bilan_avant_stockage_kWh': df_sim['bilan_horaire_kWh'].sum(),
        'energie_non_servie_kWh': df_sim['energie_non_servie_kWh'].sum(),
        'energie_ecretee_kWh': df_sim['energie_ecretee_kWh'].sum(),
        'soc_min_pct': df_sim['etat_batterie_pct'].min(),
        'soc_max_pct': df_sim['etat_batterie_pct'].max(),
    }
    return df_sim, summary

df_year_sim, annual_summary = simulate_storage(
    df_sizing,
    annual_panel_count,
    annual_battery_count,
    initial_soc=max_soc,
)

annual_csv_path = f'simulation_annuelle_{annual_panel_count}p_{annual_battery_count}b.csv'
df_year_sim.to_csv(annual_csv_path, sep=';', index=False)
display(pd.DataFrame([annual_summary]))
print(f"CSV simulation annuelle: {annual_csv_path}")

In [ ]:
# Graphiques annuels: consommation, generation et energie stockee.
import matplotlib.pyplot as plt

df_year_plot = df_year_sim.set_index('local_time').sort_index()
daily_energy = df_year_plot.resample('D').agg({
    'consommation_kW': 'sum',
    'generation_totale_kWh': 'sum',
    'energie_stockee_batteries_kWh': ['min', 'mean', 'max'],
    'energie_non_servie_kWh': 'sum',
    'energie_ecretee_kWh': 'sum',
})
daily_energy.columns = [
    'consommation_kWh',
    'generation_kWh',
    'batterie_min_kWh',
    'batterie_moyenne_kWh',
    'batterie_max_kWh',
    'energie_non_servie_kWh',
    'energie_ecretee_kWh',
]

total_capacity_kWh = annual_battery_count * battery_capacity_kWh
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(daily_energy.index, daily_energy['consommation_kWh'], label='Consommation', color='tab:red')
axes[0].plot(daily_energy.index, daily_energy['generation_kWh'], label='Generation vent + solaire', color='tab:green')
axes[0].set_ylabel('kWh/jour')
axes[0].set_title(f'Energie journaliere - {annual_panel_count} panneaux / {annual_battery_count} batteries')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

axes[1].plot(daily_energy.index, daily_energy['batterie_moyenne_kWh'], label='Stock moyen', color='tab:blue')
axes[1].fill_between(daily_energy.index, daily_energy['batterie_min_kWh'], daily_energy['batterie_max_kWh'], color='tab:blue', alpha=0.18, label='Min-max journalier')
axes[1].axhline(min_soc * total_capacity_kWh, color='black', linestyle='--', linewidth=1, label='Limite 20%')
axes[1].axhline(max_soc * total_capacity_kWh, color='black', linestyle=':', linewidth=1, label='Limite 90%')
axes[1].set_ylabel('kWh stockes')
axes[1].set_title('Energie stockee dans les batteries')
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

axes[2].plot(daily_energy.index, daily_energy['energie_ecretee_kWh'], label='Energie ecretee', color='tab:orange')
axes[2].plot(daily_energy.index, daily_energy['energie_non_servie_kWh'], label='Energie non servie', color='tab:purple')
axes[2].set_ylabel('kWh/jour')
axes[2].set_title('Energie perdue ou non servie')
axes[2].legend(loc='upper left')
axes[2].grid(True, alpha=0.3)

fig.tight_layout()
annual_plot_path = 'plot_annee_energie_batterie_generation.png'
fig.savefig(annual_plot_path, dpi=180, bbox_inches='tight')
plt.show()
print(f"Graphique annuel: {annual_plot_path}")

In [ ]:
# Journees representatives par saison, choisies comme les jours les plus proches du profil median saisonnier.
season_order = ['Hiver', 'Printemps', 'Ete', 'Automne']
season_by_month = {
    12: 'Hiver', 1: 'Hiver', 2: 'Hiver',
    3: 'Printemps', 4: 'Printemps', 5: 'Printemps',
    6: 'Ete', 7: 'Ete', 8: 'Ete',
    9: 'Automne', 10: 'Automne', 11: 'Automne',
}

df_year_sim['date'] = df_year_sim['local_time'].dt.floor('D')
df_year_sim['season'] = df_year_sim['local_time'].dt.month.map(season_by_month)
daily_profiles = df_year_sim.groupby('date').agg(
    season=('season', 'first'),
    consommation_kWh=('consommation_kW', 'sum'),
    generation_kWh=('generation_totale_kWh', 'sum'),
    batterie_moyenne_kWh=('energie_stockee_batteries_kWh', 'mean'),
    energie_non_servie_kWh=('energie_non_servie_kWh', 'sum'),
).reset_index()

representative_days = []
features = ['consommation_kWh', 'generation_kWh', 'batterie_moyenne_kWh']
for season in season_order:
    season_days = daily_profiles[daily_profiles['season'] == season].copy()
    med = season_days[features].median()
    scale = season_days[features].std().replace(0, 1)
    distance = (((season_days[features] - med) / scale) ** 2).sum(axis=1)
    selected_day = season_days.loc[distance.idxmin()]
    representative_days.append(selected_day)

representative_days_df = pd.DataFrame(representative_days)
display(representative_days_df[['season', 'date', 'consommation_kWh', 'generation_kWh', 'batterie_moyenne_kWh', 'energie_non_servie_kWh']])

fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=True)
axes = axes.flatten()
for ax, (_, row) in zip(axes, representative_days_df.iterrows()):
    day = row['date']
    day_data = df_year_sim[df_year_sim['date'] == day].copy()
    hour = day_data['local_time'].dt.hour

    ax.plot(hour, day_data['consommation_kW'], label='Consommation', color='tab:red')
    ax.plot(hour, day_data['generation_totale_kWh'], label='Generation', color='tab:green')
    ax.set_title(f"{row['season']} - {day.date()}")
    ax.set_ylabel('kWh/h')
    ax.set_xticks(range(0, 24, 3))
    ax.grid(True, alpha=0.3)

    ax_battery = ax.twinx()
    ax_battery.plot(hour, day_data['energie_stockee_batteries_kWh'], label='Batterie', color='tab:blue', alpha=0.8)
    ax_battery.set_ylabel('kWh stockes')

    lines, labels = ax.get_legend_handles_labels()
    lines_b, labels_b = ax_battery.get_legend_handles_labels()
    ax.legend(lines + lines_b, labels + labels_b, loc='upper left', fontsize=8)

fig.supxlabel('Heure')
fig.tight_layout()
season_plot_path = 'plot_journees_typiques_saisons.png'
fig.savefig(season_plot_path, dpi=180, bbox_inches='tight')
plt.show()
print(f"Graphique saisons: {season_plot_path}")